# Interpolation and the Runge warning

A polynomial of degree $n - 1$ through $n$ distinct inputs is an interpolant: the residual sum of squares is zero. Matching the sample does not keep the polynomial close to a smooth function between the inputs, or beyond them. The function $1/(1 + x^2)$ makes the arithmetic exact.


## An even interpolant

At $x = -2, -1, 0, 1, 2$,

$$
\dfrac{1}{1 + x^2} = \dfrac{1}{5},\ \dfrac{1}{2},\ 1,\ \dfrac{1}{2},\ \dfrac{1}{5}.
$$

The function is even, so the interpolating polynomial of degree at most $4$ has only even powers. The conditions at $x = 0, 1, 2$ determine it:

$$
\begin{aligned}
b_0 &= 1, \\
b_0 + b_2 + b_4 &= \dfrac{1}{2}, \\
b_0 + 4b_2 + 16b_4 &= \dfrac{1}{5}.
\end{aligned}
$$

The last two become $b_2 + b_4 = -1/2$ and $b_2 + 4b_4 = -1/5$. Subtract:

$$
3b_4 = -\dfrac{1}{5} + \dfrac{1}{2} = \dfrac{3}{10} \Longrightarrow b_4 = \dfrac{1}{10}, \qquad
b_2 = -\dfrac{1}{2} - \dfrac{1}{10} = -\dfrac{3}{5}.
$$

So

$$
p(x) = 1 - \dfrac{3}{5}x^2 + \dfrac{1}{10}x^4.
$$

It matches $1/(1 + x^2)$ at the five nodes by construction. The odd coefficients are zero because the target values are symmetric.


## Between the nodes, and past them

At $x = 1/2$, which lies between $0$ and $1$,

$$
\begin{aligned}
p\!\left(\dfrac{1}{2}\right)
&= 1 - \dfrac{3}{5}\cdot\dfrac{1}{4} + \dfrac{1}{10}\cdot\dfrac{1}{16}
= 1 - \dfrac{3}{20} + \dfrac{1}{160}
= \dfrac{137}{160}, \\[4pt]
\dfrac{1}{1 + (1/2)^2} &= \dfrac{4}{5} = \dfrac{128}{160}.
\end{aligned}
$$

The interpolant is high by $9/160$. At $x = 3/2$,

$$
p\!\left(\dfrac{3}{2}\right) = \dfrac{5}{32}, \qquad
\dfrac{1}{1 + (3/2)^2} = \dfrac{4}{13}, \qquad
p - \dfrac{1}{1 + x^2} = -\dfrac{63}{416}.
$$

The interpolant is now low. At $x = 3$, outside the nodes,

$$
p(3) = 1 - \dfrac{3}{5}\cdot 9 + \dfrac{1}{10}\cdot 81 = 1 - \dfrac{27}{5} + \dfrac{81}{10} = \dfrac{37}{10},
$$

while $1/(1 + 9) = 1/10$. The function $1/(1 + x^2)$ always lies in $(0, 1]$. The polynomial that matches it at five nodes equals $3.7$ at $x = 3$. Interpolation has no residual at the nodes with which to warn about that.


In [1]:
# The even interpolant matches at the nodes and misses between and beyond them.
from fractions import Fraction
def p(x):
    return 1 - Fraction(3, 5) * x ** 2 + Fraction(1, 10) * x ** 4

def target(x):
    return 1 / (1 + x ** 2)

nodes = [Fraction(x) for x in (-2, -1, 0, 1, 2)]
print("node gaps", [p(x) - target(x) for x in nodes])
print("at 1/2", p(Fraction(1, 2)), target(Fraction(1, 2)))
print("at 3/2", p(Fraction(3, 2)), target(Fraction(3, 2)))
print("at 3", p(Fraction(3)), target(Fraction(3)))
assert all(p(x) == target(x) for x in nodes)
assert p(Fraction(1, 2)) - target(Fraction(1, 2)) == Fraction(9, 160)
assert p(Fraction(3, 2)) == Fraction(5, 32)
assert target(Fraction(3, 2)) == Fraction(4, 13)
assert p(Fraction(3)) == Fraction(37, 10)
assert target(Fraction(3)) == Fraction(1, 10)


node gaps [Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1), Fraction(0, 1)]
at 1/2 137/160 4/5
at 3/2 5/32 4/13
at 3 37/10 1/10


## One change: a quadratic least-squares fit does not interpolate

Degree $2$ on these five values cannot match every node. By evenness the slope is zero, and the normal equations for $b_0 + b_2 x^2$ use $\sum x^2 = 10$ and $\sum x^4 = 34$.         The right-hand side is

$$
\sum y = \dfrac{1}{5} + \dfrac{1}{2} + 1 + \dfrac{1}{2} + \dfrac{1}{5} = \dfrac{12}{5}, \qquad
\sum x^2 y = 4\cdot\dfrac{1}{5} + \dfrac{1}{2} + \dfrac{1}{2} + 4\cdot\dfrac{1}{5} = \dfrac{13}{5}.
$$

Clear the denominators in $5b_0 + 10b_2 = 12/5$ and $10b_0 + 34b_2 = 13/5$:

$$
25b_0 + 50b_2 = 12, \qquad 50b_0 + 170b_2 = 13.
$$

Subtract twice the first from the second: $70b_2 = -11$, so $b_2 = -11/70$. Then $25b_0 = 139/7$ and

$$
b_0 = \dfrac{139}{175}, \qquad b_2 = -\dfrac{11}{70}.
$$

The residuals at $x = -2, -1, 0, 1, 2$ are $6/175$, $-24/175$, $36/175$, $-24/175$, $6/175$, and

$$
\mathrm{RSS} = \dfrac{72}{875}.
$$

Least squares spends the missing flexibility as residuals at the nodes. Interpolation spends it as oscillation between them.


In [2]:
# The even quadratic least-squares fit is 139/175 - (11/70) x^2.
from fractions import Fraction
x = [-2, -1, 0, 1, 2]
y = [Fraction(1, 5), Fraction(1, 2), 1, Fraction(1, 2), Fraction(1, 5)]
b0, b2 = Fraction(139, 175), Fraction(-11, 70)
residuals = [yi - (b0 + b2 * xi ** 2) for xi, yi in zip(x, y)]
rss = sum(r ** 2 for r in residuals)
print("residuals", residuals)
print("RSS", rss)
assert residuals == [
    Fraction(6, 175),
    Fraction(-24, 175),
    Fraction(36, 175),
    Fraction(-24, 175),
    Fraction(6, 175),
]
assert sum(residuals) == 0
assert sum(xi ** 2 * r for xi, r in zip(x, residuals)) == 0
assert rss == Fraction(72, 875)


residuals [Fraction(6, 175), Fraction(-24, 175), Fraction(36, 175), Fraction(-24, 175), Fraction(6, 175)]
RSS 72/875


## Pitfall

A residual sum of squares equal to zero is a statement about the nodes only. $p(3) = 37/10$ has residual zero at $x \in \{-2,-1,0,1,2\}$ and is a poor description of $1/(1+x^2)$ at $x = 3$. Choosing the degree $n - 1$ because it minimizes the residual sum will always choose interpolation when the inputs are distinct. The minimum is zero for a reason that has nothing to do with the function between the data.


In [3]:
# Zero residuals at the nodes, and a value outside (0, 1] one step past the last node.
from fractions import Fraction
print("p(3)", Fraction(37, 10))
print("target range ends at", 1)
assert Fraction(37, 10) > 1


p(3) 37/10
target range ends at 1


## Summary

- The degree-$4$ interpolant of $1/(1+x^2)$ on these nodes is $1 - (3/5)x^2 + (1/10)x^4$.
- At $x = 1/2$ it is high by $9/160$. At $x = 3$ it equals $37/10$, while the function equals $1/10$.
- The least-squares quadratic is $139/175 - (11/70)x^2$, with $\mathrm{RSS} = 72/875$. It does not pass through every node.
